# Task 2 — Face Image Processing
## Data Preprocessing & Machine Learning — Formative Assessment 2
### GROUP 9

**Subsystem:** Face Image Collection, Augmentation & Feature Extraction

---
### What this notebook does
| Step | Description |
|------|-------------|
| 1 | Import image-processing libraries (OpenCV, scikit-image) |
| 2 | Scan the `data/images/` folder for collected face photos |
| 3 | Preview collected images (neutral, smiling, surprised) |
| 4 | Apply **augmentations**: horizontal flip, rotate ±15°, grayscale |
| 5 | Extract **histogram features** (RGB colour histograms) |
| 6 | Save extracted features to `data/processed/image_features.csv` |
| 7 | Visualise augmented examples for verification |

> **Setup instructions for teammates:**  
> 1. Create a subfolder `data/images/<your_name>/`  
> 2. Add at least 3 face photos (`.jpg` or `.png`)  
> 3. Run all cells in order  
> 4. Confirm `image_features.csv` is saved  

> **Running on Google Colab?** Install packages with the pip line in Cell 1, then upload images.

## Step 1 — Import Libraries

`opencv-python` provides image loading, colour conversion, and augmentation utilities.  
`scikit-image` offers additional signal-processing-based feature extractors.  
A graceful fallback is included — if OpenCV is missing, the notebook prints guidance and skips processing.

In [1]:
# Uncomment on Google Colab:
# !pip install opencv-python scikit-image pandas numpy matplotlib -q

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    import cv2
    from skimage import exposure
    CV2_AVAILABLE = True
    print("OpenCV version:", cv2.__version__)
except ImportError:
    CV2_AVAILABLE = False
    print("OpenCV not installed.")
    print("Install with:  pip install opencv-python scikit-image")
    print("Notebook will continue but image operations will be skipped.")

# Paths
ROOT     = Path('/content') if Path('/content').exists() else Path.cwd().resolve().parents[0]
IMG_DIR  = ROOT / 'data' / 'images'
OUT_DIR  = ROOT / 'data' / 'processed'
PLOTS_DIR= ROOT / 'outputs' / 'plots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Image directory: {IMG_DIR}  (exists: {IMG_DIR.exists()})")

OpenCV not installed.
Install with:  pip install opencv-python scikit-image
Notebook will continue but image operations will be skipped.
Image directory: C:\Users\user\Desktop\kayonga-elvis task-formative2\data\images  (exists: True)


*▸ **Output above:** OpenCV version is printed if available. The image directory path is confirmed.  
If `exists: False`, create the directory and add your face photos before re-running.*

## Step 2 — Discover & Inspect Collected Images

Recursively search `data/images/` for all `.jpg` and `.png` files.  
For each image found, print the file path and display a preview.

In [2]:
image_paths = list(IMG_DIR.rglob('*.jpg')) + list(IMG_DIR.rglob('*.png'))
print(f"Total images found: {len(image_paths)}")

if not image_paths:
    print("\nNo images found. Please add face photos to data/images/<your_name>/")
    print("Expected structure:")
    print("  data/")
    print("  └── images/")
    print("      └── <your_name>/")
    print("          ├── neutral.jpg")
    print("          ├── smiling.jpg")
    print("          └── surprised.jpg")
else:
    for p in image_paths[:6]:
        print(f"  {p}")

Total images found: 0

No images found. Please add face photos to data/images/<your_name>/
Expected structure:
  data/
  └── images/
      └── <your_name>/
          ├── neutral.jpg
          ├── smiling.jpg
          └── surprised.jpg


*▸ **Output above:** Lists up to 6 discovered image paths. Each teammate should see their own subfolder listed here.  
If no images are found, follow the setup instructions in the notebook header before continuing.*

## Step 3 — Preview Face Images

Display a grid of collected images (up to 6) to visually verify they loaded correctly.  
Images are converted from BGR (OpenCV default) to RGB for matplotlib display.

In [3]:
if not CV2_AVAILABLE or not image_paths:
    print("Skipping preview — OpenCV not available or no images found.")
else:
    n_preview = min(6, len(image_paths))
    fig, axes = plt.subplots(1, n_preview, figsize=(3 * n_preview, 4))
    if n_preview == 1:
        axes = [axes]
    for ax, path in zip(axes, image_paths[:n_preview]):
        img_bgr = cv2.imread(str(path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (128, 128))
        ax.imshow(img_resized)
        ax.set_title(path.parent.name + '/' + path.name[:12], fontsize=8)
        ax.axis('off')
    fig.suptitle('Collected Face Images Preview', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '10_face_preview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Displayed {n_preview} images.")

Skipping preview — OpenCV not available or no images found.


*▸ **Output above:** A grid of face photos displayed at 128×128 pixels.  
Verify that:  
- Images are clear and faces are visible  
- Folder names match the expected teammate structure  
- No corrupted / blank images appear*

## Step 4 — Image Augmentation

Augmentation artificially increases the diversity of the training set by applying controlled transformations.  
For face images, we apply:

| Augmentation | Description |
|---|---|
| **Horizontal flip** | Mirror the face left-right |
| **Rotation ±15°** | Slight tilt to simulate head angle variation |
| **Grayscale** | Remove colour information (tests texture-only features) |

Each original image generates 3 augmented variants, tripling the dataset size.

In [4]:
def augment_image(img_rgb):
    """Return list of augmented variants of a given RGB image."""
    augmented = []
    # 1. Horizontal flip
    flipped = cv2.flip(img_rgb, 1)
    augmented.append(('flip', flipped))
    # 2. Rotation +15 degrees
    h, w = img_rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), 15, 1.0)
    rotated_pos = cv2.warpAffine(img_rgb, M, (w, h))
    augmented.append(('rot+15', rotated_pos))
    # 3. Rotation -15 degrees
    M2 = cv2.getRotationMatrix2D((w//2, h//2), -15, 1.0)
    rotated_neg = cv2.warpAffine(img_rgb, M2, (w, h))
    augmented.append(('rot-15', rotated_neg))
    return augmented

if not CV2_AVAILABLE or not image_paths:
    print("Skipping augmentation — OpenCV not available or no images found.")
else:
    # Preview augmentations for the first image
    demo_img_bgr = cv2.imread(str(image_paths[0]))
    demo_img_rgb = cv2.cvtColor(demo_img_bgr, cv2.COLOR_BGR2RGB)
    demo_img_rgb = cv2.resize(demo_img_rgb, (128, 128))

    variants = augment_image(demo_img_rgb)
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    axes[0].imshow(demo_img_rgb); axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')
    for ax, (name, img) in zip(axes[1:], variants):
        ax.imshow(img); ax.set_title(name, fontweight='bold'); ax.axis('off')
    fig.suptitle('Augmentation Preview — One Image', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '11_face_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Each image generates {len(variants)} augmented variants.")
    print(f"Total training images after augmentation: {len(image_paths) * (1 + len(variants))}")

Skipping augmentation — OpenCV not available or no images found.


*▸ **Output above:** Shows the original image alongside all three augmented variants.  
Verify that:  
- The flip is a mirror image (not upside-down)  
- Rotations are subtle (±15°, not 90°)  
The bottom line tells you the total training set size after augmentation.*

## Step 5 — Feature Extraction (Colour Histograms)

For each image (and its augmented variants), we extract a **colour histogram feature vector**:  
- Split the image into 3 channels (R, G, B)  
- Compute a 16-bin histogram per channel  
- Concatenate → **48-dimensional feature vector** per image

This compact representation captures the colour distribution of the face photo without storing raw pixel arrays.

In [5]:
def extract_hist_features(rgb_img):
    """Extract a 48-dim colour histogram feature vector from an RGB image."""
    hist_r = cv2.calcHist([rgb_img], [0], None, [16], [0, 256]).flatten()
    hist_g = cv2.calcHist([rgb_img], [1], None, [16], [0, 256]).flatten()
    hist_b = cv2.calcHist([rgb_img], [2], None, [16], [0, 256]).flatten()
    # Normalise to [0, 1] so values are scale-invariant
    total_pixels = rgb_img.shape[0] * rgb_img.shape[1]
    return np.concatenate([hist_r, hist_g, hist_b]) / total_pixels

if not CV2_AVAILABLE or not image_paths:
    print("OpenCV not available — creating empty features CSV template.")
    cols = ['image_path', 'member_name'] + [f'feat_{i}' for i in range(48)]
    pd.DataFrame(columns=cols).to_csv(OUT_DIR / 'image_features.csv', index=False)
    print(f"Empty template saved to {OUT_DIR / 'image_features.csv'}")
else:
    records = []
    for img_path in image_paths:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"Warning: could not read {img_path}")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (128, 128))
        member_name = img_path.parent.name

        # Original
        feat = extract_hist_features(img_rgb)
        row = {'image_path': str(img_path), 'member_name': member_name, 'augmentation': 'original'}
        row.update({f'feat_{i}': float(feat[i]) for i in range(len(feat))})
        records.append(row)

        # Augmented
        for aug_name, aug_img in augment_image(img_rgb):
            feat = extract_hist_features(aug_img)
            row = {'image_path': str(img_path), 'member_name': member_name, 'augmentation': aug_name}
            row.update({f'feat_{i}': float(feat[i]) for i in range(len(feat))})
            records.append(row)

    features_df = pd.DataFrame(records)
    features_df.to_csv(OUT_DIR / 'image_features.csv', index=False)
    print(f"Extracted features for {len(image_paths)} images + augmentations = {len(records)} rows")
    print(f"Feature CSV shape: {features_df.shape}")
    print(f"Saved to: {OUT_DIR / 'image_features.csv'}")
    display(features_df.head(3))

OpenCV not available — creating empty features CSV template.
Empty template saved to C:\Users\user\Desktop\kayonga-elvis task-formative2\data\processed\image_features.csv


*▸ **Output above:** Each row in `image_features.csv` corresponds to one image (original or augmented).  
Columns: `image_path`, `member_name`, `augmentation`, then `feat_0` through `feat_47`.  
This CSV is consumed by the multimodal integration notebook (`Task4_Multimodal_Integration.ipynb`).*